# Hubble Agent v2: Tools, Routing, and Grounded Answers

This notebook rebuilds the Hubble agent from *Building Your First LLM-Based AI Agent* into a real **agent**. Three things change:

| | v1 (pipeline) | v2 (agent) |
|---|---|---|
| Control flow | Fixed: classify → fetch → answer | The LLM **decides** which tools to call and when to stop |
| Facts | The LLM guesses the country from raw coordinates | A **deterministic tool** looks up land/ocean, country and continent |
| Routing | `int(llm_output)` inside a `try/except` | **Structured output** + a **conditional edge** |

The guiding principle: **tools for facts, the LLM for language and decisions.**

Graph we will build:

```
START → route_question ──(off_topic)──→ refuse → END
              │
         (on topic)
              ↓
            agent ⇄ tools        (loop until the LLM stops calling tools)
              ↓
             END
```

This notebook is the Claude's modification of the simple variant (v1) without a loop.

## Setup

In [ ]:
!pip install -q langgraph langchain-openai reverse_geocoder global-land-mask pycountry-convert skyfield

In [ ]:
# OpenRouter key: in Colab, add it under "Secrets" (key icon) as OPENROUTER_TOKEN.
try:
    from google.colab import userdata
    OPENROUTER_TOKEN = userdata.get("OPENROUTER_TOKEN")
except ImportError:
    import os, getpass
    OPENROUTER_TOKEN = os.environ.get("OPENROUTER_TOKEN") or getpass.getpass("OpenRouter token: ")

### Choose a model that supports tool calling

The agent loop only works if the model can emit **tool calls**. On openrouter.ai/models, filter by *Supported parameters → tools*. If the model you pick does not support tools, `bind_tools` will fail or the model will ignore the tools. Both of those failures make a good debugging exercise.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-ultra-550b-a55b:free",   # any tool-capable model
    temperature=0,
)

## Step 1: Tools

A **tool** is an ordinary Python function with a docstring and type hints. LangChain turns the signature and docstring into a schema the LLM can read. That is why the docstrings below are written **for the model**: they say when to call the tool and what it returns.

### Tool 1: Where is Hubble right now?

Instead of a third-party API with a key, we compute the position ourselves. We download Hubble's latest **TLE** (two-line orbital elements) from CelesTrak and propagate it with the SGP4 model in `skyfield`. This is the same method the tracking websites use.

In [ ]:
from datetime import timezone
from langchain_core.tools import tool
from skyfield.api import load, wgs84

HUBBLE_NORAD_ID = 20580
TLE_URL = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={HUBBLE_NORAD_ID}&FORMAT=tle"
ts = load.timescale()

@tool
def get_hubble_position() -> dict:
    """Return the current sub-satellite point of the Hubble Space Telescope.

    Call this whenever you need to know where Hubble is right now.
    Returns latitude and longitude in degrees, altitude in km, and the UTC time.
    """
    hubble = load.tle_file(TLE_URL, filename="hubble.tle", reload=True)[0]
    t = ts.now()
    geocentric = hubble.at(t)
    lat, lon = wgs84.latlon_of(geocentric)
    return {
        "latitude": round(lat.degrees, 4),
        "longitude": round(lon.degrees, 4),
        "altitude_km": round(wgs84.height_of(geocentric).km, 1),
        "utc_time": t.utc_datetime().astimezone(timezone.utc).isoformat(timespec="seconds"),
    }

get_hubble_position.invoke({})   # tools can be tested on their own, without any LLM

### Tool 2: What is at these coordinates?

This tool works in three stages, each a deterministic lookup:
1. `global_land_mask` tells us whether a point is land or water. It uses a 1 km grid built from GEBCO bathymetry data.
2. Over land, `reverse_geocoder` finds the nearest populated place (offline, using GeoNames data). That gives us the country, and `pycountry_convert` maps the country to its continent.
3. Over water, a small rule-based function assigns the ocean basin.

> **Limitations worth discussing in class:**
> - The "nearest town" heuristic can pick the neighboring country right at a border.
> - The ocean boundaries are hand-drawn approximations.
>
> The key difference from v1 is that both are *known, inspectable* errors, not hallucinations.

In [ ]:
def ocean_name(lat: float, lon: float) -> str:
    """Approximate ocean basin for a point already known to be over water."""
    lon = ((lon + 180) % 360) - 180          # normalize to [-180, 180)
    if lat >= 66:
        return "Arctic Ocean"
    if lat <= -60:
        return "Southern Ocean"
    # Indian Ocean: east of Africa (20°E), west of Indonesia/Australia
    if lat < 30 and 20 <= lon <= 100:
        return "Indian Ocean"
    if lat <= -10 and 100 < lon <= 147:
        return "Indian Ocean"
    # Atlantic: between the Americas and Europe/Africa (incl. Gulf of Mexico,
    # Caribbean and Mediterranean); the western edge follows the coastline.
    if lat > 30:
        west = -100
    elif lat > 18:
        west = -98                            # Gulf of Mexico
    elif lat >= 8:
        west = -77 - (lat - 8) * 1.8          # Central American isthmus
    else:
        west = -68                            # Pacific coast of South America
    east = 45 if lat > 30 else 20
    if west < lon < east:
        return "Atlantic Ocean"
    return "Pacific Ocean"

In [ ]:
import reverse_geocoder as rg
import pycountry_convert as pc
from global_land_mask import globe

CONTINENTS = {"AF": "Africa", "AN": "Antarctica", "AS": "Asia", "EU": "Europe",
              "NA": "North America", "OC": "Oceania", "SA": "South America"}

@tool
def describe_location(latitude: float, longitude: float) -> dict:
    """Describe what lies under a point on Earth's surface.

    Use this to find out which country, continent, or ocean a latitude/longitude
    pair is over. Returns a dict with 'surface' ("land" or "water") and then
    either 'country', 'continent', 'nearest_place' (land) or 'ocean' (water).
    """
    if not globe.is_land(latitude, longitude):
        return {"surface": "water", "ocean": ocean_name(latitude, longitude)}

    place = rg.search((latitude, longitude), mode=1, verbose=False)[0]
    cc = place["cc"]
    try:
        country = pc.country_alpha2_to_country_name(cc)
        continent = CONTINENTS[pc.country_alpha2_to_continent_code(cc)]
    except KeyError:                     # e.g. codes not in the ISO table
        country, continent = cc, "unknown"
    return {"surface": "land", "country": country, "continent": continent,
            "nearest_place": f"{place['name']}, {place['admin1']}"}

# Sanity checks -- know your tool before you hand it to an LLM
for pt in [(27.84, -27.22), (48.85, 2.35), (-1.29, 36.82), (0.0, -140.0)]:
    print(pt, describe_location.invoke({"latitude": pt[0], "longitude": pt[1]}))

## Step 2: Routing with structured output

v1 asked the model to "write only a single number" and then parsed it with `int()`. That approach is fragile. A model that answers `"1."` or `"The answer is 1"` silently becomes class 0.

With `with_structured_output`, we give the model a **schema** (a Pydantic class), and LangChain returns a validated Python object. The `Literal` type restricts the model to three allowed labels.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Route(BaseModel):
    """Classify a user question for the Hubble location agent."""
    kind: Literal["country", "continent_or_ocean", "off_topic"] = Field(
        description="'country' if the user asks which country Hubble is over; "
                    "'continent_or_ocean' if they ask about a continent or ocean; "
                    "'off_topic' for anything else.")

router = llm.with_structured_output(Route)

router.invoke("Which ocean is the Hubble telescope over right now?")

## Step 3: State and nodes

We use `MessagesState`, a state type whose `messages` list is **appended to** rather than overwritten. This message list is the agent's working memory: it stores the user's question, every tool call, and every tool result. We add one extra field for the route.

Each node returns **only the keys it changes**. LangGraph merges those updates into the state.

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage
from langgraph.graph import MessagesState

class HubbleState(MessagesState):
    question_kind: str

tools = [get_hubble_position, describe_location]
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = SystemMessage(content=(
    "You answer questions about where the Hubble Space Telescope currently is. "
    "Never guess locations: always call get_hubble_position, then describe_location "
    "with the coordinates it returns. Base your answer only on the tool results. "
    "Reply in one or two sentences and include the coordinates."
))

def route_question(state: HubbleState):
    question = state["messages"][-1].content
    return {"question_kind": router.invoke(question).kind}

def refuse(state: HubbleState):
    return {"messages": [AIMessage(content=(
        "I can only tell you which country, continent, or ocean Hubble is "
        "currently over. Please ask a question like that."))]}

def agent(state: HubbleState):
    # one system message only -- some providers reject several
    system = SystemMessage(content=SYSTEM_PROMPT.content +
                           f" The user is asking about: {state['question_kind']}.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

## Step 4: Wire the graph

There are two kinds of **conditional edges** here:
- `route_after_classification` is a function we write. It reads the state and returns the name of the next node.
- `tools_condition` is a prebuilt function. It sends the flow to `"tools"` if the last AI message contains tool calls, and to `END` otherwise.

The edge `tools → agent` closes the **loop**. After every tool result, the LLM decides what to do next.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def route_after_classification(state: HubbleState):
    return "refuse" if state["question_kind"] == "off_topic" else "agent"

builder = StateGraph(HubbleState)
builder.add_node("route_question", route_question)
builder.add_node("refuse", refuse)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "route_question")
builder.add_conditional_edges("route_question", route_after_classification, ["agent", "refuse"])
builder.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
builder.add_edge("tools", "agent")
builder.add_edge("refuse", END)

graph = builder.compile()
graph

## Step 5: Run it, and inspect the trace

`recursion_limit` caps the number of steps. This is a safety net in case the model keeps calling tools forever.

Printing every message shows the trace: what the model decided, which tools it called with which arguments, and what the tools returned. **Always check that the final answer matches the tool output.**

In [ ]:
def ask(question: str):
    result = graph.invoke({"messages": [("user", question)]},
                          config={"recursion_limit": 10})
    for m in result["messages"]:
        m.pretty_print()
    return result

_ = ask("Which country is Hubble flying over?")

In [ ]:
_ = ask("Which ocean or continent is Hubble above right now?")

In [ ]:
_ = ask("How many years did Leo Tolstoy live?")   # should be refused without calling any tool

## Exercises (ideas for the final project)

1. **Evaluate grounding.** Run the agent 10 times. For each run, extract the last `describe_location` tool result and check automatically that the final answer mentions the same country or ocean. Report the agreement rate.
2. **Add a tool.** Write `next_pass_over(city: str)`, which tells you when Hubble next passes over a city (`skyfield` provides `find_events`). Do you need to change the graph, or only the tool list and the system prompt? Explain why.
3. **Break it on purpose.** Remove "Never guess locations" from the system prompt and use a smaller model. How often does it answer without calling tools? What does this tell you about prompt design versus graph design?
4. **Add memory.** Compile with `checkpointer=MemorySaver()` and pass a `thread_id`, so that a follow-up like "and five minutes later?" works.
5. **Improve a tool.** Replace the rule-based `ocean_name` with a polygon lookup (for example, Natural Earth marine areas with `geopandas`), and measure how often the two methods disagree on random water points.